# mspas


In [ ]:
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, BooleanType, StringType, TimestampType,
)

PAIS_ISO3 = "GTM"
CIE10_RE  = r"^[A-Z][0-9]{2}[0-9A-Z]?$"

SCHEMA_MORTALIDAD_GENERAL = {
    "id":          LongType(),
    "pais_iso3":   StringType(),
    "anio":        IntegerType(),
    "defunciones": LongType(),
    "source_table":StringType(),
    "ingestion_ts":TimestampType(),
}

SCHEMA_TASA = {
    "id":            LongType(),
    "pais_iso3":     StringType(),
    "anio":          IntegerType(),
    "tasa_por_100k": DoubleType(),
    "source_table":  StringType(),
    "ingestion_ts":  TimestampType(),
}


def _verify_schema(df: DataFrame, expected: dict, table_name: str) -> None:
    actual = {f.name: f.dataType for f in df.schema.fields}
    errors = []
    for col, exp_type in expected.items():
        if col not in actual:
            errors.append(f"  MISSING  '{col}'")
        elif type(actual[col]) != type(exp_type):
            errors.append(f"  MISMATCH '{col}': expected {exp_type}, got {actual[col]}")
    extra = [c for c in actual if c not in expected]
    if extra:
        errors.append(f"  EXTRA    {extra}")
    if errors:
        print(f"[SCHEMA] {table_name} — {len(errors)} problema(s):")
        for msg in errors:
            print(msg)
    else:
        print(f"[SCHEMA] {table_name} — OK ({len(actual)} columnas, tipos correctos)")


def _normalize_col_names(df: DataFrame) -> DataFrame:
    for col_name in df.columns:
        normalized = col_name.strip().lower().replace(" ", "_").replace("-", "_")
        if normalized != col_name:
            df = df.withColumnRenamed(col_name, normalized)
    if "year" in df.columns:
        df = df.withColumnRenamed("year", "anio")
    return df


def _replace_ignorado(df: DataFrame) -> DataFrame:
    return (
        df
        .replace("Ignorado", None)
        .replace("ignorado", None)
        .replace("IGNORADO", None)
    )


def _validate_cie10(df: DataFrame, col: str) -> DataFrame:
    return df.withColumn(col,
        F.when(
            F.col(col).isNotNull()
            & F.upper(F.trim(F.col(col))).rlike(CIE10_RE),
            F.upper(F.trim(F.col(col)))
        ).otherwise(F.lit(None).cast(StringType()))
    )


def _cast_year(df: DataFrame, col: str, lo: int, hi: int) -> DataFrame:
    return df.withColumn(col,
        F.when(F.col(col).cast(IntegerType()).between(lo, hi),
               F.col(col).cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )


def _cast_count(df: DataFrame, col: str) -> DataFrame:
    return df.withColumn(col,
        F.when(F.col(col).cast(LongType()) >= 0, F.col(col).cast(LongType()))
         .otherwise(F.lit(None).cast(LongType()))
    )


def _cast_rate(df: DataFrame, col: str) -> DataFrame:
    return df.withColumn(col,
        F.when(F.col(col).cast(DoubleType()) >= 0, F.col(col).cast(DoubleType()))
         .otherwise(F.lit(None).cast(DoubleType()))
    )


def _add_metadata(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("pais_iso3", F.lit(PAIS_ISO3))
        .withColumn("id", F.monotonically_increasing_id().cast(LongType()))
        .withColumnRenamed("_source",              "source_table")
        .withColumnRenamed("_ingestion_timestamp", "ingestion_ts")
    )


def _normalize_cols(df: DataFrame, rename_map: dict, null_defaults: dict) -> DataFrame:
    select_exprs = []
    seen = set()
    for c in df.columns:
        target = rename_map.get(c, c)
        if target not in seen:
            select_exprs.append(F.col(f"`{c}`").alias(target))
            seen.add(target)
    df = df.select(*select_exprs)
    for col, dtype in null_defaults.items():
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None).cast(dtype))
    return df


def transform_mortalidad_general(df: DataFrame) -> DataFrame:
    df = _normalize_col_names(df)
    df = _replace_ignorado(df)
    df = _normalize_cols(df,
        rename_map={"total_deaths": "defunciones"},
        null_defaults={"defunciones": LongType()},
    )
    df = _cast_year(df, "anio", 2000, 2030)
    df = _cast_count(df, "defunciones")
    df = df.dropDuplicates()
    df = _add_metadata(df)
    return df.select("id", "pais_iso3", "anio", "defunciones", "source_table", "ingestion_ts")


def transform_tasa(df: DataFrame) -> DataFrame:
    df = _normalize_col_names(df)
    df = _replace_ignorado(df)
    df = _normalize_cols(df,
        rename_map={"mortality_rate_per_1000": "tasa_por_100k"},
        null_defaults={"tasa_por_100k": DoubleType()},
    )
    # fuente en por-1000; schema en por-100k
    df = df.withColumn("tasa_por_100k", F.col("tasa_por_100k").cast(DoubleType()) * 100)
    df = _cast_year(df, "anio", 2000, 2020)
    df = _cast_rate(df, "tasa_por_100k")
    df = df.dropDuplicates()
    df = _add_metadata(df)
    return df.select("id", "pais_iso3", "anio", "tasa_por_100k", "source_table", "ingestion_ts")


spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

try:
    raw_general    = spark.read.table("semi2.sandbox.raw_mspas_mortalidad_general")
    staged_general = transform_mortalidad_general(raw_general)
    _verify_schema(staged_general, SCHEMA_MORTALIDAD_GENERAL, "stage.mspas_mortalidad_general")
    display(staged_general)
    staged_general.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("stage.mspas_mortalidad_general")

    raw_tasa    = spark.read.table("semi2.sandbox.raw_mspas_tasa")
    staged_tasa = transform_tasa(raw_tasa)
    _verify_schema(staged_tasa, SCHEMA_TASA, "stage.mspas_tasa")
    display(staged_tasa)
    staged_tasa.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("stage.mspas_tasa")

except Exception as e:
    print(f"Error al procesar sandbox.raw_mspas_*: {e}")
    raise
